### Tóm tắt cách vận hành:
- Lần chạy 1: Đặt OFFSET = 0, BATCH_SIZE = 35000. Bấm Save & Run All, tắt máy.
- Lần chạy 2 (sau khi lần 1 xong hoặc chạy song song trên Notebook khác): Đặt OFFSET = 35000, BATCH_SIZE = 35000.

Bộ đếm Time Guard (10.8 tiếng) đảm bảo dù mạng có chậm đến mấy thì máy chủ Kaggle luôn dừng kịp để đóng gói zip ra tab Output, không bao giờ bị lỗi quá 12h!

Mỗi lần chạy xong sẽ tạo ra một file zip riêng biệt: corpus_b_small_offset_0.zip, corpus_b_small_offset_35000.zip... sẵn sàng trong tab Output để bạn tải về bất kỳ lúc nào!

In [ ]:
# ==============================================================================
# R2AI 2026 - RUNNER TẬN DỤNG 100% CÁC HÀM TRONG SRC/CORPUS_CRAWLER
# Cào 49 domain nhỏ + Time Guard 10h + Checkpoint SQLite + ShardWriter chuẩn
# ==============================================================================

# 1. Cài đặt các thư viện cần thiết (nếu chạy trên Kaggle/Colab)
%pip install -q trafilatura zstandard pyarrow datasets aiohttp protego langdetect nest_asyncio

In [ ]:
import os
import sys
import time
import asyncio
from pathlib import Path
from dataclasses import replace

# Tự động nạp thư mục src vào sys.path để import corpus_crawler
CRAWLER_DIR = Path("tasks/01_corpus_collection/crawler").resolve()
if (CRAWLER_DIR / "src").exists():
    sys.path.insert(0, str(CRAWLER_DIR / "src"))
elif Path("src").exists():
    sys.path.insert(0, str(Path("src").resolve()))

# Hỗ trợ chạy asyncio trực tiếp trong Jupyter Notebook mà không bị RuntimeError
import nest_asyncio
nest_asyncio.apply()

# ------------------------------------------------------------------------------
# 2. IMPORT CÁC MODULE VÀ HÀM CHÍNH TỪ SRC/CORPUS_CRAWLER
# ------------------------------------------------------------------------------
from corpus_crawler.settings import Settings, load_settings
from corpus_crawler.state import CrawlState
from corpus_crawler.output import ShardWriter
from corpus_crawler.fetch import RespectfulFetcher
from corpus_crawler.extract import make_records
from corpus_crawler.text import normalize_hostname

import aiohttp
from datasets import load_dataset
import pyarrow as pa
import pyarrow.parquet as pq

# ==============================================================================
# 3. CẤU HÌNH THÔNG SỐ PHÂN LÔ & TIME GUARD
# ==============================================================================
OWNER = "b"
OFFSET = 0                        # Thay đổi sau mỗi lần chạy (0, 35000, 70000...)
BATCH_SIZE = 35000                # Số URL cào trong phiên này
MAX_RUNTIME_HOURS = 10.0          # Time Guard: dừng an toàn sau đúng 10 tiếng

START_TIME = time.time()
MAX_RUNTIME_SECONDS = MAX_RUNTIME_HOURS * 3600

# Thư mục dữ liệu gốc của dự án
CORPUS_ROOT = Path("corpus").resolve()
(CORPUS_ROOT / "viz" / "docs" / OWNER).mkdir(parents=True, exist_ok=True)
(CORPUS_ROOT / "viz" / "crawl_log" / OWNER).mkdir(parents=True, exist_ok=True)
(CORPUS_ROOT / "viz" / "manifest" / OWNER).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# 4. DANH SÁCH 49-51 DOMAIN NHỎ CỦA MEMBER B (ĐÃ LOẠI 2 DOMAIN KHỦNG)
# ------------------------------------------------------------------------------
DOMAINS_B_SMALL = frozenset({
    "a-hospital.com", "suckhoecongdongonline.vn", "suckhoedoisong.vn", "thanhnien.vn",
    "wujue.com", "baby.39.net", "vinmec.com", "medlatec.vn", "jbk.39.net",
    "giadinhonline.vn", "baonghean.vn", "care.39.net", "baodanang.vn", "fitness.39.net",
    "hellobacsi.com", "food.39.net", "fk.39.net", "cancer.39.net", "jb39.com",
    "vov2.vov.vn", "phuyen.baodaklak.vn", "heart.39.net", "suckhoeviet.org.vn",
    "pmc-ecm-healthblog.beta.pharmacity.io", "nhandan.vn", "gan.39.net", "wei.39.net",
    "baidianfeng.familydoctor.com.cn", "vietnamplus.vn", "shen.39.net", "tienphong.vn",
    "thaythuocvietnam.vn", "bachmai.gov.vn", "dantri.com.vn", "gk.39.net",
    "baolangson.vn", "gc.39.net", "baophutho.vn", "benhviennhitrunguong.gov.vn",
    "sj.39.net", "baothanhhoa.vn", "pharmacity.vn", "familydoctor.cn",
    "qy.familydoctor.com.cn", "v.familydoctor.com.cn", "ask.familydoctor.com.cn",
    "mega.vietnamplus.vn", "yanglao.familydoctor.com.cn", "fk.99.com.cn",
    "special.vietnamplus.vn", "ypk.familydoctor.com.cn"
})

# ==============================================================================
# 5. KHỞI TẠO ĐỐI TƯỢNG SETTINGS CỦA CORPUS_CRAWLER
# ==============================================================================
config_file = CRAWLER_DIR / "config.toml"
settings = load_settings(config_file if config_file.exists() else None)

# Ghi đè cấu hình để chạy riêng cho 49 domain nhỏ của Member B
settings = replace(
    settings,
    owner=OWNER,
    corpus_root=CORPUS_ROOT,
    domains=DOMAINS_B_SMALL,
    global_concurrency=32,
    per_domain_concurrency=1,
    requests_per_domain=1.0,
    shard_docs=20000,
    progress_seconds=5,
    user_agent="R2AI2026CorpusResearch/1.0 (Respectful multi-domain crawler - Member B)"
)

# Khởi tạo CrawlState (Database SQLite WAL để checkpoint chống mất dữ liệu)
state_db_path = CORPUS_ROOT / "viz" / f"state-{OWNER}-small.sqlite3"
state = CrawlState(state_db_path)

# ==============================================================================
# 6. TẢI DỮ LIỆU TỪ HUGGING FACE VÀ NẠP VÀO CRAWLSTATE THEO BATCH
# ==============================================================================
if not state.has_registered_urls():
    print("[*] Đang tải corpus từ Hugging Face: AIGuruTinix/ViBioMIR...")
    hf_dataset = load_dataset("AIGuruTinix/ViBioMIR", "corpus", split="train")
    
    print("[*] Đang lọc URL thuộc 49 domain nhỏ...")
    all_small_urls = []
    for row in hf_dataset:
        d = normalize_hostname(row["url"])
        if d in DOMAINS_B_SMALL:
            all_small_urls.append((int(row["id"]), str(row["url"]), str(d)))
            
    print(f"[✓] Tổng số URL từ các domain nhỏ: {len(all_small_urls):,}")
    
    # Cắt phân lô theo OFFSET & BATCH_SIZE
    target_batch = all_small_urls[OFFSET : OFFSET + BATCH_SIZE]
    print(f"[*] Nạp batch #{OFFSET:,} -> #{OFFSET + len(target_batch):,} vào State DB...")
    
    # Nạp vào database checkpoint bằng hàm nội bộ của state.py
    state._insert_source_batch(target_batch)
    print(f"[✓] Đã nạp thành công {len(target_batch):,} URLs vào {state_db_path.name}")
else:
    print(f"[*] Tiếp tục từ State DB đã có: {state_db_path.name}")

total_urls, done_urls, pending_urls = state.totals()
print(f"-> Tổng URL batch: {total_urls:,} | Đã hoàn thành: {done_urls:,} | Còn lại: {pending_urls:,}")

# Khởi tạo ShardWriter chuẩn từ src/corpus_crawler/output.py
writer = ShardWriter(settings, state)

# ==============================================================================
# 7. ENGINE CHẠY ĐỒNG THỜI KÈM TIME GUARD 10 GIỜ (TẬN DỤNG FETCH.PY & EXTRACT.PY)
# ==============================================================================
async def run_batch_with_timeguard():
    connector = aiohttp.TCPConnector(limit=settings.global_concurrency)
    timeout = aiohttp.ClientTimeout(total=settings.timeout_seconds)
    headers = {"User-Agent": settings.user_agent}
    
    stop_event = asyncio.Event()
    last_report = time.monotonic()
    
    # Sử dụng RespectfulFetcher chính gốc từ src/corpus_crawler/fetch.py
    # (Đã tích hợp sẵn Protego đọc robots.txt và lock giãn cách 1s/domain)
    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=headers) as session:
        fetcher = RespectfulFetcher(settings, session)

        async def worker():
            nonlocal last_report
            while not stop_event.is_set():
                # KIỂM TRA TIME GUARD (10 TIẾNG)
                if (time.time() - START_TIME) >= MAX_RUNTIME_SECONDS:
                    print(f"\n[!] TIME GUARD kích hoạt: Đã đạt mốc {MAX_RUNTIME_HOURS} tiếng! Dừng mềm để đóng gói an toàn.")
                    stop_event.set()
                    return

                # Lấy URL tiếp theo từ hàng đợi SQLite bằng state.claim_next()
                row = state.claim_next()
                if row is None:
                    if state.pending_count() == 0:
                        return
                    await asyncio.sleep(0.05)
                    continue

                # 1. Tải trang bằng fetcher.fetch() từ src/fetch.py
                result = await fetcher.fetch(str(row["url"]), str(row["domain"]))

                # 2. Bóc tách và chuẩn hóa bằng make_records() từ src/extract.py
                log_record, doc_record = await asyncio.to_thread(
                    make_records,
                    int(row["id"]),
                    str(row["url"]),
                    str(row["domain"]),
                    result,
                    settings,
                )

                # 3. Lưu kết quả và dedupe bằng state.save_result() từ src/state.py
                state.save_result(int(row["id"]), result.attempts, log_record, doc_record)

                # 4. Xuất shard khi đủ số lượng bằng writer.export_ready() từ src/output.py
                writer.export_ready()

                # Báo cáo tiến độ ra màn hình
                now = time.monotonic()
                if now - last_report >= settings.progress_seconds:
                    total, done, pending = state.totals()
                    elapsed = max(time.time() - START_TIME, 0.001)
                    rate = done / elapsed
                    eta_m = (pending / rate / 60) if rate > 0 else 0
                    print(
                        f"\rTiến độ: {done:,}/{total:,} ({done/total*100:.1f}%) | "
                        f"{rate:.1f} URL/s | Còn lại: {pending:,} | ETA: {eta_m:.1f} phút",
                        end="",
                        flush=True
                    )
                    last_report = now

        # Chạy 32 workers song song
        workers = [asyncio.create_task(worker()) for _ in range(settings.global_concurrency)]
        try:
            await asyncio.gather(*workers)
        except Exception as e:
            stop_event.set()
            raise e
        finally:
            # Ép đóng shard dở dang và ghi manifest đầy đủ
            writer.export_ready(force=True)

# ------------------------------------------------------------------------------
# 8. THỰC THI VÀ ĐÓNG GÓI ZIP OUTPUT
# ------------------------------------------------------------------------------
print("\n[*] Khởi động tiến trình cào dữ liệu...")
try:
    asyncio.run(run_batch_with_timeguard())
finally:
    state.close()

print("\n\n" + "="*70)
print(f"[✓] CÀO DỮ LIỆU HOÀN TẤT CHO BATCH #{OFFSET}!")
print(f"• Thống kê trạng thái: {state.completed_counts()}")

# Tự động nén toàn bộ thư mục corpus thành file zip theo batch
zip_filename = f"/kaggle/working/corpus_b_small_offset_{OFFSET}.zip"
print(f"[*] Đang nén kết quả thành: {zip_filename}...")
os.system(f"zip -r -q {zip_filename} corpus/")
print(f"[✓] HOÀN TẤT AN TOÀN! Tệp '{zip_filename}' đã sẵn sàng trong thư mục làm việc.")